In [1]:
# cell 1 — set up imports & paths
import sys
from pathlib import Path
import json
import pandas as pd

# -------------- 修改为你本地的项目根路径 --------------
PROJECT_ROOT = Path("/Users/shiyiliu/Desktop/Latent_Space_Vis").resolve()
# --------------------------------------------------------

# 插入 scripts/GNNGMMAnalyzer 到 sys.path（使 embedding_pipeline_core 可导入）
GNNGMM_DIR = PROJECT_ROOT / "scripts" / "GNNGMMAnalyzer"
if str(GNNGMM_DIR) not in sys.path:
    sys.path.insert(0, str(GNNGMM_DIR))

# 如果 backend 里有 util 也需要，可以加入 backend 目录
BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GNNGMM_DIR on sys.path:", GNNGMM_DIR.exists())
print("BACKEND_DIR on sys.path:", BACKEND_DIR.exists())

# 现在导入后端模块（和 backend/app.py 使用的同样函数）
from embedding_pipeline_core import load_embeddings_from_files
from embedding_pipeline import EmbeddingPipeline

print("Modules imported OK")

PROJECT_ROOT: /Users/shiyiliu/Desktop/Latent_Space_Vis
GNNGMM_DIR on sys.path: True
BACKEND_DIR on sys.path: True
Modules imported OK


In [2]:
# cell 2 — load embeddings and run pipeline
import torch   # 若 load_embeddings_from_files 内使用 torch
from pathlib import Path

# Embeddings 目录 —— 改为你本地路径（最好用绝对路径）
EMB_DIR = PROJECT_ROOT / "backend" / "anyembeddings"   # example: /Users/.../Latent_Space_Vis/anyembeddings

# 可选：如果你想指定具体 layer 文件与名字（与你 frontend 使用的一致）
layer_files = ["emb_unique_cas.pt","output0_unique_cas.pt","output1_unique_cas.pt"]
layer_names = ["emb_unique_cas","output0_unique_cas","output1_unique_cas"]
layer_files_dict = dict(zip(layer_names, layer_files))

print("Embeddings dir exists:", EMB_DIR.exists())
if not EMB_DIR.exists():
    raise FileNotFoundError(f"Embeddings dir not found: {EMB_DIR}")

# 调用 load_embeddings_from_files
# 如果你的 load_embeddings_from_files 接口支持只传目录也可以删掉第二个参数
embeddings = load_embeddings_from_files(EMB_DIR, layer_files_dict)
# embeddings 应该是 dict: { layer_name: numpy/tensor array (n_samples, dim) }
print("Loaded layers:", list(embeddings.keys()))
for k, v in embeddings.items():
    try:
        print(k, getattr(v, "shape", (len(v),)))
    except Exception:
        print(k, type(v))

# 构造 metadata（可选）
metadata = {"note": "notebook run"}

# 创建 pipeline（参考 backend/app.py 的用法）
tmp_output_dir = PROJECT_ROOT / "frontend" / "public" / "data" / "tmp_notebook"
tmp_output_dir.mkdir(parents=True, exist_ok=True)

pipeline = EmbeddingPipeline(
    embeddings=embeddings,
    output_dir=tmp_output_dir,
    id_mapping=None,    # 如果有 id map 的 json path，可加载并传入 dict
    n_components=4,     # 你想降维成多少维
    metadata=metadata
)

# 运行 GMM + MDS（和 backend 一样）
print("Fitting GMMs...")
pipeline.fit_gmms()
print("Computing MDS / distance embedding...")
pipeline.compute_mds()

# 生成 scatter + sankey DataFrame
scatter_df = pipeline.generate_scatter_plot_data()
sankey_df = pipeline.generate_sankey_data()

print("scatter rows:", len(scatter_df))
print("sankey rows:", len(sankey_df))

# 简单查看前几行
display(scatter_df.head())
display(sankey_df.head())

Embeddings dir exists: True
  ✓ emb_unique_cas: shape (152, 128)
  ✓ output0_unique_cas: shape (152, 256)
  ✓ output1_unique_cas: shape (152, 256)
Loaded layers: ['emb_unique_cas', 'output0_unique_cas', 'output1_unique_cas']
emb_unique_cas (152, 128)
output0_unique_cas (152, 256)
output1_unique_cas (152, 256)
Using 4 GMM components
Fitting GMMs...

Fitting GMMs for all layers...
  Fitting GMM for emb_unique_cas...
  Fitting GMM for output0_unique_cas...
  Fitting GMM for output1_unique_cas...
✓ Fitted GMMs for 3 layers
Computing MDS / distance embedding...

Computing MDS positions...
✓ Computed MDS positions for 3 layers

Generating scatter plot data...
✓ Generated scatter plot data: 456 points

Generating Sankey diagram data...
✓ Generated Sankey diagram data: 304 transitions
scatter rows: 456
sankey rows: 304


,id,x,y,color_value,layer
0,0,1053.377761,1234.786399,0.0,emb_unique_cas
1,1,1557.396823,921.892890,0.0,emb_unique_cas
2,2,910.397481,-2696.177409,0.0,emb_unique_cas
3,3,3084.250371,813.262800,0.0,emb_unique_cas
4,4,1441.918031,-237.268487,0.0,emb_unique_cas


,real_id,from_layer,to_layer,from_cluster,to_cluster
0,0,emb_unique_cas,output0_unique_cas,0,3
1,1,emb_unique_cas,output0_unique_cas,0,3
2,2,emb_unique_cas,output0_unique_cas,2,3
3,3,emb_unique_cas,output0_unique_cas,1,3
4,4,emb_unique_cas,output0_unique_cas,2,3


In [4]:
# cell 3 — save results to disk for later使用或前端读取
OUT_DIR = tmp_output_dir
scatter_csv = OUT_DIR / "scatter_notebook.csv"
sankey_csv = OUT_DIR / "sankey_notebook.csv"

scatter_df.to_csv(scatter_csv, index=False)
sankey_df.to_csv(sankey_csv, index=False)

# 如果你更喜欢 json（scatter 通常是 records）
scatter_json = OUT_DIR / "scatter_notebook.json"
sankey_json = OUT_DIR / "sankey_notebook.json"
scatter_df.to_json(scatter_json, orient="records")
sankey_df.to_json(sankey_json, orient="records")

print("Saved scatter ->", scatter_csv)
print("Saved sankey  ->", sankey_csv)

Saved scatter -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/scatter_notebook.csv
Saved sankey  -> /Users/shiyiliu/Desktop/Latent_Space_Vis/frontend/public/data/tmp_notebook/sankey_notebook.csv


In [30]:
# from IPython.display import IFrame

# IFrame(
#     src="http://localhost:5173/?data=/tmp_notebook/sankey_notebook.json",
#     width="100%",
#     height=600
# )

In [45]:
# Cell 1 — 内联 bundle（只需在会话开始或更改 bundle 后运行一次），并挂载第一个 sankey
from IPython.display import HTML, display
from pathlib import Path
import json

# prepare data for first sankey (可以替换为任意 DataFrame 列表 / 子集)
data1 = sankey_df.head(200).to_json(orient="records")

# read the built IIFE bundle (确保路径正确)
bundle_path = Path("dist/notebook-sankey.js")
if not bundle_path.exists():
    raise FileNotFoundError(f"Bundle not found at {bundle_path}. 请先运行 npm run build 并确认路径。")
bundle_code = bundle_path.read_text()

html = f"""
<div id="sankey-root-1" style="height:500px; border:1px solid #ddd; margin-bottom:8px;"></div>

<!-- 注入第一个数据 -->
<script>window.__SANKEY_DATA__ = {data1};</script>

<!-- minimal polyfill for `process` if some deps reference it -->
<script>
  if (typeof window.process === 'undefined') {{
    window.process = {{ env: {{}} }};
  }}
  if (typeof globalThis.process === 'undefined') {{
    globalThis.process = window.process;
  }}
</script>

<!-- inline bundle (IIFE) -->
<script>
{bundle_code}
</script>

<!-- 调用 mountApp 来挂载第一个图 -->
<script>
  try {{
    if (window.mountApp) {{
      // explicit initialData ensures this mount uses data1 (doesn't rely on window.__SANKEY_DATA__ later)
      window.mountApp("sankey-root-1", {{ initialData: {data1} }});
    }} else {{
      console.error("mountApp 未找到，请检查 bundle 是否正确内联并且导出了 window.mountApp");
    }}
  }} catch (e) {{
    console.error("mountApp 调用失败:", e);
  }}
</script>
"""

display(HTML(html))

In [47]:
# Cell: 定义 render_sankey() 工具（一次性加载 bundle，然后多次调用渲染）
from IPython.display import HTML, display
from pathlib import Path
import json, uuid

DIST_BUNDLE_PATH = Path("dist/notebook-sankey.js")  # 若你的 bundle 在别处，改这里

def render_sankey(df_or_json, elem_id=None, height=480, bundle_path=DIST_BUNDLE_PATH):
    """
    Render a Sankey in the notebook.
    - df_or_json: pandas.DataFrame or list/dict (JSON-serializable) or JSON string
    - elem_id: optional element id; if None a unique id is created
    - height: container height in px
    - bundle_path: path to IIFE bundle (default 'dist/notebook-sankey.js')
    """
    # ensure elem_id
    if elem_id is None:
        elem_id = f"sankey-{uuid.uuid4().hex[:8]}"

    # prepare JSON string for data
    if hasattr(df_or_json, "to_json"):
        data_json = df_or_json.to_json(orient="records")
    else:
        # if user passed list/dict or string
        if isinstance(df_or_json, str):
            # assume it's already a JSON string
            data_json = df_or_json
        else:
            data_json = json.dumps(df_or_json)

    # make sure bundle exists for first-time inline; if missing we still try fallback
    bundle_exists = bundle_path.exists()

    # Build HTML that:
    # 1) creates a container div for the sankey
    # 2) if bundle not loaded yet, inline the bundle + register window.__renderSankey
    # 3) then call window.__renderSankey with the provided data and elem_id
    # Note: we inject a minimal process polyfill before executing bundle to avoid `process is not defined`.
    html_parts = []

    html_parts.append(f'<div id="{elem_id}" style="height:{height}px; border:1px solid #eee; margin:6px 0;"></div>')

    # Inline helper + bundle only if not loaded on the page yet.
    # We'll attempt to detect via window.__NOTEBK_SANK_LOADED flag.
    # The inline script will set that flag.
    if bundle_exists:
        bundle_code = bundle_path.read_text()
        inline_bootstrap = f"""
<script>
(function() {{
  // if already initialized, don't re-inline bundle (just call render)
  if (window.__NOTEBK_SANK_LOADED) {{
    return;
  }}

  // minimal runtime polyfill for process if some deps reference it
  if (typeof window.process === 'undefined') {{
    window.process = {{ env: {{}} }};
  }}
  if (typeof globalThis.process === 'undefined') {{
    globalThis.process = window.process;
  }}

  // inline the IIFE bundle now
  try {{
    {bundle_code}
  }} catch (e) {{
    console.error("Error executing inlined Sankey bundle:", e);
    return;
  }}

  // Now the bundle should have exposed window.mountApp or window.NotebookSankey.mountApp
  // Provide a small helper that normalizes mount invocation and stores a registry of mounts
  window.__NOTEBK_SANK_LOADED = true;
  window.__renderSankey_calls = window.__renderSankey_calls || [];

  window.__renderSankey = function(data, elemId, opts) {{
    // if mountApp is not available, try to look under NotebookSankey
    var mount = window.mountApp || (window.NotebookSankey && window.NotebookSankey.mountApp);
    if (!mount) {{
      console.error("mountApp not found after bundle execution. Make sure the bundle exports mountApp to window.");
      return null;
    }}
    // call mount with explicit initialData to avoid relying on global single value
    var handle = mount(elemId, {{ initialData: data, dataUrl: opts && opts.dataUrl }});
    // store handle for debugging/unmount
    window.__renderSankey_calls.push({{elemId: elemId, handle: handle}});
    return handle;
  }};
}})();
</script>
"""
        html_parts.append(inline_bootstrap)
    else:
        # if bundle missing, still create a small loader that will try to fetch via Jupyter's /files/ path
        inline_loader = f"""
<script>
(function() {{
  if (window.__NOTEBK_SANK_LOADED) return;
  if (typeof window.process === 'undefined') {{
    window.process = {{ env: {{}} }};
  }}
  if (typeof globalThis.process === 'undefined') {{
    globalThis.process = window.process;
  }}

  try {{
    // try synchronous XHR read from Jupyter files URL as fallback
    var xhr = new XMLHttpRequest();
    xhr.open('GET', '/files/{bundle_path.as_posix()}', false);
    xhr.send(null);
    if (xhr.status === 200) {{
      var script = document.createElement('script');
      script.type = 'text/javascript';
      script.text = xhr.responseText;
      document.head.appendChild(script);
      // expose helper as above
      window.__NOTEBK_SANK_LOADED = true;
      window.__renderSankey_calls = window.__renderSankey_calls || [];
      window.__renderSankey = function(data, elemId, opts) {{
        var mount = window.mountApp || (window.NotebookSankey && window.NotebookSankey.mountApp);
        if (!mount) {{ console.error("mountApp not found after loading bundle via /files"); return null; }}
        var handle = mount(elemId, {{ initialData: data, dataUrl: opts && opts.dataUrl }});
        window.__renderSankey_calls.push({{elemId: elemId, handle: handle}});
        return handle;
      }};
    }} else {{
      console.warn('Could not load bundle via /files/; please run npm run build and ensure dist/notebook-sankey.js exists.'); 
    }}
  }} catch (e) {{
    console.warn('Exception while trying to load bundle via /files/:', e);
  }}
}})();
</script>
"""
        html_parts.append(inline_loader)

    # Finally add a small script that calls the helper with the actual data (this will work whether bundle was inlined now or had been inlined earlier)
    call_script = f"""
<script>
(function() {{
  var data = {data_json};
  var attempts = 0;
  function tryRender() {{
    if (window.__renderSankey) {{
      try {{
        window.__renderSankey(data, "{elem_id}", {{}});
      }} catch (e) {{
        console.error("renderSankey call error:", e);
      }}
    }} else {{
      attempts++;
      if (attempts < 30) {{
        // wait a bit and retry (bundle may be executing)
        setTimeout(tryRender, 100);
      }} else {{
        console.error("renderSankey helper not available after waiting; please rerun the initialization cell that inlines the bundle.");
      }}
    }}
  }}
  tryRender();
}})();
</script>
"""
    html_parts.append(call_script)

    display(HTML("\n".join(html_parts)))
    return elem_id  # return the DOM id used

In [48]:
render_sankey(sankey_df.head(120), height=450)

'my-sankey-1'

In [50]:
render_sankey(sankey_df.tail(120), height=450)

'sankey-d2bd7433'